# Modelos direto por UF · 2/4 · estatísticos e árvores com covariáveis (Fases 3 e 6)

**Antes de rodar (uma vez só)**, em `MyDrive/dengue_V4/` (que já tem `dados/`, `referencias_imdc3/` e os arquivos da competição):
1. suba **`uf_lib.py`** e **`uf_autoarima.R`** (pasta `data_imdc_2026/UF/`);
2. suba a **pasta `UF/dados/` com o nome `dados_uf/`** (5 arquivos `.parquet` + `vizinhanca_uf.npz`);
3. substitua **`comp_modelos.py`** e **`comp_inla.R`** pelas versões novas (de `COMPETICAO/`): elas trazem o INLA não linear e a correção do `val_size`.

Os resultados vão para `MyDrive/dengue_V4/resultados_uf/`. Se a sessão cair, rode de novo: o que já foi feito é pulado.
⚠️ Ao abrir vários notebooks ao mesmo tempo, **espere a célula 1 do primeiro terminar** antes de rodar os outros: sessões simultâneas podem criar duas pastas `resultados_uf` no Drive. Se isso acontecer, renomeie a cópia para `resultados_uf_2`; o `CompUF4` e o `Comp5` leem todas as pastas `resultados_uf*`.

**Temporadas:** 2018–2021 = **afinação** (é onde se escolhe tudo); 2022–2025 = **teste** (S1–S4, olhado uma vez no `CompUF4`).

| Notebook | O que roda | Ambiente | Tempo estimado |
|---|---|---|---|
| `CompUF1_portados` | os 13 modelos da competição, direto por UF | CPU High-RAM | 1–2 h (com INLA) |
| `CompUF2_CPU` | auto.arima / auto.ARIMAX (Python e R), 6 VARs, INLA não linear, dois estágios, GBMs com covariáveis, ablação | CPU High-RAM | 4–6 h |
| `CompUF3_GPU` | AGCRN, Chronos, 6 redes com pooling (com e sem exógenas), STID, 4 redes de grafo, Chronos-2 | GPU L4 | 8–12 h |
| `CompUF4_avaliacao` | escolha em 2018–2021, teste em 2022–2025, ensembles | CPU | ~15 min |

**Ambiente:** CPU *High-RAM*. Dá para dividir em sessões com `RODAR_AGORA`.

| Grupo | Modelos | Ideia |
|---|---|---|
| auto.arima | `AutoARIMA_py_UF`, `AutoARIMA_r_UF` | **tudo automático**: d (KPSS), p, q e drift pelo AICc; sazonalidade por Fourier com K = 1..6 também pelo AICc. `py` = statsforecast, `r` = o `forecast::auto.arima` original |
| auto.arima estacionário | `AutoARIMA_d0_py_UF`, `AutoARIMA_d0_r_UF` | o mesmo, mas com d = 0 e sem drift (como o vencedor da competição): impede a extrapolação de tendência por 68 semanas |
| auto.ARIMAX | `AutoARIMAX_py_UF`, `AutoARIMAX_r_UF`, `AutoARIMAX_d0_py_UF`, `AutoARIMAX_d0_r_UF` | + ENSO (persistência da SE25) e anomalias de temperatura e chuva (SEAS5 por 6 meses, depois 0) |
| VAR (várias UFs juntas) | `VAR_macro_UF`, `VAR_espacial_UF`, `BVAR_Minnesota_UF` (+ versões `X` com ENSO) | anomalias de log-incidência; *macro* = um VAR por grande região, ordem pelo BIC; *espacial* = defasagens próprias + média dos vizinhos; *BVAR* = todas as 26 UFs com encolhimento de Minnesota |
| INLA não linear | `GHR_NL_INLA_UF` | BYM2 nas 26 UFs + efeitos não lineares (RW1 nos decis) das anomalias de clima + ENSO |
| Dois estágios | `DoisEstagios_UF`, `DoisEstagios_pool_UF`, `DoisEstagios_pool_sel_UF` | GBM prevê tamanho e pico da temporada com variáveis da SE25; a curva vem das temporadas passadas; *sel* = só as variáveis aprovadas na avaliação |
| GBMs | `GBM_lgb_casos_UF`, `GBM_lgb_selecionadas_UF`, `GBM_lgb_todas_UF`, `GBM_{xgb,cat}_selecionadas_UF`, `GBM_lgb_todas_pool_UF` | quantílicos diretos (68 semanas); *casos* = sem covariáveis, *selecionadas* = conjunto padrão, *todas* = tudo; *pool* = treina também com as 435 regiões |
| Aumento de dados | `GBM_{lgb,cat}_selecionadas_aum_UF` | linhas extras com temporadas 1,5–4× maiores (Fase 6) |
| Ablação | `GBM_lgb_todas_sem_<grupo>_UF` | um grupo de variáveis a menos por vez; **só 2018–2021** |

**Comparabilidade:** todo modelo com exógenas futuras usa o MESMO conjunto `U.EXOG_FUTURAS` = ENSO (persistido da SE25) + anomalias de temperatura e chuva (SEAS5 por 6 meses, depois 0); os VARX usam só o ENSO. Cada família tem a versão **sem** exógenas, para medir o ganho.

In [ ]:
NOTEBOOK = 'uf_cpu'
PACOTES = 'statsforecast catboost'

# @title Drive, arquivos e pacotes
import os, sys, shutil, time, glob, subprocess, json, traceback
import numpy as np, pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PASTA_DRIVE, PASTA_LOCAL, NO_COLAB = '/content/drive/MyDrive/dengue_V4', '/content/uf', True
except ImportError:
    PASTA_DRIVE = PASTA_LOCAL = os.path.abspath('.'); NO_COLAB = False
if NO_COLAB:
    os.makedirs(PASTA_LOCAL, exist_ok=True)
    for item in ('v4_lib.py', 'comp_lib.py', 'comp_modelos.py', 'comp_inla.R', 'uf_lib.py', 'uf_autoarima.R', 'dados',
                 'dados_uf', 'referencias_imdc3'):
        o, d = os.path.join(PASTA_DRIVE, item), os.path.join(PASTA_LOCAL, item)
        assert os.path.exists(o), f'Faltou subir {item} para {PASTA_DRIVE}'
        shutil.copytree(o, d, dirs_exist_ok=True) if os.path.isdir(o) else shutil.copy(o, d)
    if PACOTES:
        r = subprocess.run(f'pip install -q {PACOTES}', shell=True, capture_output=True, text=True)
        print(r.stdout[-400:], r.stderr[-400:])
sys.path.insert(0, PASTA_LOCAL)
PASTA_RES = os.path.join(PASTA_DRIVE, 'resultados_uf')
PASTA_COMP = os.path.join(PASTA_DRIVE, 'resultados_competicao')
os.makedirs(os.path.join(PASTA_RES, 'graficos'), exist_ok=True)
import importlib, v4_lib as V, comp_lib as L, comp_modelos as CM, uf_lib as U
for m in (V, L, CM, U):
    importlib.reload(m)
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'nenhuma', '· CPUs:', os.cpu_count())
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
ctx = U.ContextoUF(os.path.join(PASTA_LOCAL, 'dados'), os.path.join(PASTA_LOCAL, 'referencias_imdc3'),
                   os.path.join(PASTA_LOCAL, 'dados_uf'))
TUNING, TESTE, TODAS = U.TUNING, U.TESTE, U.TODAS_UF
M = 500                                  # trajetórias por UF
REFAZER = False

In [ ]:
USAR_INLA = True   # False pula a instalação e os modelos INLA
if USAR_INLA:
    # @title Instalar o R-INLA (só na primeira vez; cópia compactada fica no Drive)
    # O Drive não deixa executar programas (o motor inla.mkl.run dá "Permission denied").
    # Por isso o INLA roda do disco local do Colab (/content/R_libs); o Drive só guarda um .tar.gz,
    # que preserva as permissões de execução.
    R_LIBS = '/content/R_libs'; os.makedirs(R_LIBS, exist_ok=True)
    os.environ['R_LIBS_USER'] = R_LIBS
    TAR_DRIVE = os.path.join(PASTA_DRIVE, 'R_libs_inla.tar.gz')
    ANTIGO_DRIVE = os.path.join(PASTA_DRIVE, 'R_libs')          # instalação antiga, direto no Drive


    def tem_inla():
        r = subprocess.run(['Rscript', '-e', 'cat(requireNamespace("INLA", quietly=TRUE))'], capture_output=True, text=True)
        return 'TRUE' in r.stdout


    def libera_execucao():
        b = os.path.join(R_LIBS, 'INLA', 'bin')
        if os.path.isdir(b):
            subprocess.run(['chmod', '-R', 'a+rx', b])


    if shutil.which('Rscript') is None:
        print('Instalando o R (~2 min)...'); subprocess.run('apt-get -qq update && apt-get -qq install -y r-base-core > /dev/null', shell=True)
    print(subprocess.run(['Rscript', '--version'], capture_output=True, text=True).stderr.split('\n')[0])
    if not tem_inla() and os.path.exists(TAR_DRIVE):
        print('Restaurando o INLA do Drive (~1 min)...'); subprocess.run(['tar', '-xzf', TAR_DRIVE, '-C', '/content'])
    if not tem_inla() and os.path.isdir(os.path.join(ANTIGO_DRIVE, 'INLA')):
        print('Copiando a instalação antiga do Drive para o disco local (~2 min)...')
        subprocess.run(f'cp -r "{ANTIGO_DRIVE}"/. "{R_LIBS}"/', shell=True)
    libera_execucao()
    if not tem_inla():
        print('Instalando o INLA (10–20 min, só na primeira vez)...')
        script = r'''
        lib <- Sys.getenv("R_LIBS_USER"); options(timeout = 3600, Ncpus = 4)
        options(repos = c(CRAN = "https://packagemanager.posit.co/cran/__linux__/jammy/latest"))
        install.packages(c("Matrix", "sp", "fmesher", "foreach", "withr", "MatrixModels"), lib = lib)
        ok <- function() requireNamespace("INLA", quietly = TRUE)
        try(install.packages("INLA", lib = lib, repos = c(getOption("repos"), INLA = "https://inla.r-inla-download.org/R/stable"), dependencies = FALSE))
        if (!ok()) try(install.packages("https://inla.r-inla-download.org/R/stable/src/contrib/INLA_25.10.19.tar.gz", lib = lib, repos = NULL, type = "source"))
        if (!ok()) try(install.packages("https://inla.r-inla-download.org/R/stable/src/contrib/INLA_25.06.07.tar.gz", lib = lib, repos = NULL, type = "source"))
        cat("INLA:", ok(), "\n")'''
        r = subprocess.run(['Rscript', '-e', script], capture_output=True, text=True)
        print(r.stdout[-1500:], r.stderr[-1500:])
        libera_execucao()
    print('INLA disponível:', tem_inla())

    # Pacotes que o INLA usa em tempo de execução ('sn' é exigido pelo inla.posterior.sample)
    EXTRAS = ['sn', 'mnormt', 'numDeriv', 'quantreg', 'SparseM', 'MatrixModels']
    falta = subprocess.run(['Rscript', '-e', 'p <- commandArgs(TRUE); cat(p[!sapply(p, requireNamespace, quietly=TRUE)])', *EXTRAS],
                           capture_output=True, text=True).stdout.split()
    if falta:
        print('Instalando pacotes auxiliares:', falta)
        subprocess.run(['Rscript', '-e', 'install.packages(commandArgs(TRUE), lib=Sys.getenv("R_LIBS_USER"), '
                        'repos="https://packagemanager.posit.co/cran/__linux__/jammy/latest")', *falta], capture_output=True)
        if os.path.exists(TAR_DRIVE):
            os.remove(TAR_DRIVE)                                  # regrava a cópia do Drive já com os auxiliares
    if tem_inla() and not os.path.exists(TAR_DRIVE):
        print('Guardando cópia compactada no Drive para as próximas sessões...')
        subprocess.run(['tar', '-czf', '/content/R_libs_inla.tar.gz', '-C', '/content', 'R_libs'])
        shutil.copy('/content/R_libs_inla.tar.gz', TAR_DRIVE)

    # Teste de fumaça: o motor do INLA precisa rodar de verdade (não basta o pacote carregar)
    teste = r'''
    suppressMessages(library(INLA)); set.seed(1)
    d <- data.frame(y = rpois(50, 5), x = rnorm(50))
    r <- inla(y ~ x, family = "poisson", data = d, inla.mode = "compact", control.compute = list(config = TRUE))
    a <- inla.posterior.sample(5, r)
    cat("MOTOR_OK", round(r$summary.fixed[1, "mean"], 2), "| amostras:", length(a), "| versão", as.character(packageVersion("INLA")), "\n")'''
    r = subprocess.run(['Rscript', '-e', teste], capture_output=True, text=True)
    print(r.stdout[-500:] if 'MOTOR_OK' in r.stdout else ('❌ O motor do INLA não rodou:\n' + r.stdout[-1500:] + r.stderr[-2500:]))
INLA_KW = dict(anos=6, M=300, threads=os.cpu_count(), r_libs=R_LIBS, pasta_script=PASTA_LOCAL) if USAR_INLA else None

In [ ]:
# @title R + pacote forecast (para o auto.arima em R; ~1 min na primeira vez)
R_LIBS = '/content/R_libs' if NO_COLAB else None
if NO_COLAB:
    os.makedirs(R_LIBS, exist_ok=True); os.environ['R_LIBS_USER'] = R_LIBS
    if shutil.which('Rscript') is None:
        print('Instalando o R (~2 min)...'); subprocess.run('apt-get -qq update && apt-get -qq install -y r-base-core > /dev/null', shell=True)
    tem = lambda: 'TRUE' in subprocess.run(['Rscript', '-e', 'cat(requireNamespace("forecast", quietly=TRUE))'],
                                           capture_output=True, text=True).stdout
    if not tem():
        r = subprocess.run(['Rscript', '-e', 'options(Ncpus=4); install.packages("forecast", lib=Sys.getenv("R_LIBS_USER"), '
                            'repos="https://packagemanager.posit.co/cran/__linux__/jammy/latest")'], capture_output=True, text=True)
        print(r.stderr[-800:])
    print('forecast disponível:', tem())
RSCRIPT = shutil.which('Rscript') or 'Rscript'
R_KW = dict(rscript=RSCRIPT, r_libs=R_LIBS, pasta_script=PASTA_LOCAL)

In [ ]:
mods = []
for est in (False, True):                                       # d livre (auto.arima puro) e d = 0 sem drift
    for ex in ((), U.EXOG_FUTURAS):                             # ARIMA e ARIMAX
        mods += [U.AutoARIMA('py', ex, estacionario=est), U.AutoARIMA('r', ex, estacionario=est, **R_KW)]
mods += [U.VARModelo(t, exog=x, M=M) for t in ('macro', 'espacial', 'bvar') for x in (False, True)]
if INLA_KW is not None:
    mods.append(U.Porta(CM.ModeloINLA('ghr_nl', **INLA_KW)))
mods += [U.DoisEstagios(pool=False, M=M), U.DoisEstagios(pool=True, M=M), U.DoisEstagios(pool=True, selecao=True, M=M),
         U.GBMDireto('lgb', None), U.GBMDireto('lgb', 'selecionadas'), U.GBMDireto('lgb', 'todas'),
         U.GBMDireto('xgb', 'selecionadas'), U.GBMDireto('cat', 'selecionadas'), U.GBMDireto('lgb', pool=True),
         U.GBMDireto('lgb', 'selecionadas', aumento=0.15), U.GBMDireto('cat', 'selecionadas', aumento=0.15)]
abl = U.ablacao('lgb')
MODELOS = {m.nome: m for m in mods + abl}
PLANO_COMPLETO = {**{m.nome: TODAS for m in mods}, **{m.nome: TUNING for m in abl}}
RODAR_AGORA = list(PLANO_COMPLETO)      # ex.: só os ARIMA numa sessão: [k for k in PLANO_COMPLETO if k.startswith('AutoARIMA')]
PLANO = {k: v for k, v in PLANO_COMPLETO.items() if k in RODAR_AGORA}
print(len(PLANO), 'modelos ·', sum(len(v) for v in PLANO.values()), 'rodadas')

In [ ]:
# @title Rodar (retoma de onde parou; um modelo com erro não interrompe os outros)
erros = []
for nome, temps in PLANO.items():
    for s in temps:
        try:
            dt = L.rodar(ctx, MODELOS[nome], s, PASTA_RES, M=M, refazer=REFAZER)
            if dt is not None:
                print(f'✅ {nome:34s} {s}: {dt / 60:5.1f} min', flush=True)
        except Exception as e:
            erros.append((nome, s, repr(e)))
            print(f'❌ {nome:34s} {s}: {repr(e)[:400]}', flush=True)
            traceback.print_exc(limit=2)
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
print('Erros:', len(erros))
pd.DataFrame(erros, columns=['modelo', 'temporada', 'erro']).to_csv(os.path.join(PASTA_RES, f'erros_{NOTEBOOK}.csv'), index=False)
t = L.tempos(PASTA_RES)
t[t.modelo.isin(list(PLANO))].groupby('modelo').segundos.agg(['count', 'mean', 'sum']).div([1, 60, 60]).round(1) \
    .rename(columns={'count': 'temporadas', 'mean': 'min/temporada', 'sum': 'min total'})